<a href="https://colab.research.google.com/github/Nahom32/Cow-Anomaly-Detection/blob/main/notebooks/Cow_Localization_and_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00


In [2]:
!pip install torch torchvision pytorchvideo decord

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 20.9 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=d42a1e255ea1a27102fa7c37d42bf7b138547e14848b106d63b7f5ea1883d654
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=826cbfd36c98c5d6384aaf2245aaec57931138cf61d14ea25d693eba7f4497b2
  Stored in directory: /root/.cache/pip/wheel

In [3]:
!pip install opencv-python tqdm pandas

In [4]:
!pip install kagglehub

In [5]:
import kagglehub

path = kagglehub.dataset_download("fandaoerji/cbvd-5cow-behavior-video-dataset")
print(path)
import os

for d in os.listdir(path):
    print(d)

100%|██████████| 10.8G/10.8G [02:04<00:00, 93.3MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11
labelframes
CBVD-5.csv
minilabelframes
videos_add
annotations
rawframes_mini
videos
miniannotations
labelframes_add


In [6]:
print("Dataset root path:", path)


print("\nDataset structure preview:")
for root, dirs, files in os.walk(path):
    if files:
        rel = os.path.relpath(root, path)
        print(f"{rel}/  → {len(files)} files (first 3: {files[:3]})")


DATA_ROOT = path
ANNOTATIONS_DIR = os.path.join(path, "annotations")
VIDEO_DIR = os.path.join(path, "video_cut")
RAWFRAMES_DIR = os.path.join(path, "rawframes")

print(f"\n Paths ready:")
print(f"   DATA_ROOT       = {DATA_ROOT}")
print(f"   ANNOTATIONS_DIR = {ANNOTATIONS_DIR}")
print(f"   VIDEO_DIR       = {VIDEO_DIR}")

Dataset root path: /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11

Dataset structure preview:
./  → 1 files (first 3: ['CBVD-5.csv'])
labelframes/labelframes/  → 4122 files (first 3: ['156_00003.jpg', '258_00002.jpg', '275_00007.jpg'])
minilabelframes/  → 4122 files (first 3: ['156_00003.jpg', '258_00002.jpg', '275_00007.jpg'])
videos_add/videos/  → 200 files (first 3: ['869.mp4', '728.mp4', '820.mp4'])
annotations/  → 11 files (first 3: ['ava_test_excluded_timestamps_v2.1.csv', 'ava_dense_proposals_val.FAIR.recall_93.9.pkl', 'ava_action_list_v2.1_for_activitynet_2018.pbtxt'])
rawframes_mini/681/  → 300 files (first 3: ['img_00271.jpg', 'img_00208.jpg', 'img_00104.jpg'])
rawframes_mini/388/  → 300 files (first 3: ['img_00271.jpg', 'img_00208.jpg', 'img_00104.jpg'])
rawframes_mini/213/  → 300 files (first 3: ['img_00271.jpg', 'img_00208.jpg', 'img_00104.jpg'])
rawframes_mini/277/  → 300 files (first 3: ['img_00271.jpg', 'img_00208.jpg', 'img_0010

In [7]:
from ultralytics import YOLO
from scipy.spatial.distance import cdist
import shutil
from pathlib import Path


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [8]:
print(f"\n Paths ready:")
print(f"   DATA_ROOT       = {DATA_ROOT}")
print(f"   ANNOTATIONS_DIR = {ANNOTATIONS_DIR}")
print(f"   VIDEO_DIR       = {VIDEO_DIR}")
DATA_ROOT = path

ANNOTATIONS_DIR = os.path.join(DATA_ROOT, "miniannotations")
FRAMES_DIR = os.path.join(DATA_ROOT, "rawframes_mini")

print(os.listdir(ANNOTATIONS_DIR))
ann_file = None

# Prioritize 'ava_train_v2.1.csv' as it seems to contain actual data
if 'ava_train_v2.1.csv' in os.listdir(ANNOTATIONS_DIR):
    ann_file = os.path.join(ANNOTATIONS_DIR, 'ava_train_v2.1.csv')
else:
    for f in os.listdir(ANNOTATIONS_DIR):
        if f.endswith(".csv"):
            ann_file = os.path.join(ANNOTATIONS_DIR, f)
            break
print("Using:", ann_file)
import pandas as pd

df = pd.read_csv(ann_file, header=None, dtype={0: str})

df.columns = [
    "video_id",
    "timestamp",
    "x1", "y1", "x2", "y2",
    "action_id",
    "target_id"
]

print(df.head())
print("Total samples:", len(df))
print("Sample frame folders:")
print(os.listdir(FRAMES_DIR)[:20])
print("Sample video_ids from annotations:")
print(df["video_id"].head(20).tolist())
valid_ids = set(os.listdir(FRAMES_DIR))

print("Number of frame folders:", len(valid_ids))
df["video_id"] = df["video_id"].apply(lambda x: str(int(x)))

df = df[df["video_id"].isin(valid_ids)]

print("Filtered dataset size:", len(df))


 Paths ready:
   DATA_ROOT       = /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11
   ANNOTATIONS_DIR = /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11/annotations
   VIDEO_DIR       = /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11/video_cut
['ava_test_excluded_timestamps_v2.1.csv', 'ava_dense_proposals_val.FAIR.recall_93.9.pkl', 'ava_action_list_v2.1_for_activitynet_2018.pbtxt', 'ava_val_v2.1.csv', 'labelmap.txt', 'ava_dense_proposals_test.FAIR.recall_93.9.pkl', 'ava_dense_proposals_train.FAIR.recall_93.9.pkl', 'ava_val_excluded_timestamps_v2.1.csv', 'ava_train_excluded_timestamps_v2.1.csv', 'ava_test_v2.1.csv', 'ava_train_v2.1.csv']
Using: /root/.cache/kagglehub/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset/versions/11/miniannotations/ava_train_v2.1.csv
  video_id  timestamp        x1        y1        x2        y2  action_id  \
0      618          2  0.4570

In [9]:
import torch
from torch.utils.data import Dataset
import os, cv2, random
import numpy as np

class CowDatasetHybrid(Dataset):
    def __init__(self, df, frames_dir, normal_classes, num_frames=8, fps=25):
        self.df = df.reset_index(drop=True)
        self.frames_dir = frames_dir
        self.num_frames = num_frames
        self.fps = fps

        self.normal_df = self.df[self.df["action_id"].isin([c+1 for c in normal_classes])]
        self.anomaly_df = self.df[~self.df["action_id"].isin([c+1 for c in normal_classes])]

    def __len__(self):
        return len(self.normal_df)

    def _load_clip(self, row):
        video_id = str(int(row.video_id))
        video_folder = os.path.join(self.frames_dir, video_id)

        if not os.path.exists(video_folder):
            return torch.zeros((self.num_frames, 3, 224, 224))

        frame_files = sorted(os.listdir(video_folder))
        if len(frame_files) == 0:
            return torch.zeros((self.num_frames, 3, 224, 224))

        center_idx = int(row.timestamp * self.fps)
        center_idx = min(max(center_idx, 0), len(frame_files)-1)

        start = max(0, center_idx - self.num_frames // 2)
        selected = frame_files[start:start+self.num_frames]

        frames = []
        for f in selected:
            img = cv2.imread(os.path.join(video_folder, f))
            if img is None:
                continue

            h, w, _ = img.shape
            x1 = int(row.x1 * w)
            y1 = int(row.y1 * h)
            x2 = int(row.x2 * w)
            y2 = int(row.y2 * h)

            x1, y1 = max(0,x1), max(0,y1)
            x2, y2 = min(w,x2), min(h,y2)

            if x2 <= x1 or y2 <= y1:
                continue

            img = img[y1:y2, x1:x2]
            img = cv2.resize(img, (224, 224))

            img = img / 255.0
            img = (img - 0.5) / 0.5

            frames.append(img)

        if len(frames) == 0:
            frames = [np.zeros((224,224,3))]*self.num_frames

        if len(frames) < self.num_frames:
            frames += [frames[-1]]*(self.num_frames-len(frames))

        frames = torch.tensor(frames).permute(0,3,1,2).float()
        return frames

    def __getitem__(self, idx):
        normal_row = self.normal_df.iloc[idx]
        normal_clip = self._load_clip(normal_row)

        # pseudo anomaly: sample from anomaly set
        pseudo_row = self.anomaly_df.sample(1).iloc[0]
        pseudo_clip = self._load_clip(pseudo_row)

        return normal_clip, pseudo_clip

In [10]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

OUTPUT_DIR = "/content/cow_detection_dataset"
FPS = 25

# Clear existing to avoid conflicts
import shutil
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(f"{OUTPUT_DIR}/train/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/train/labels", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/val/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/val/labels", exist_ok=True)

# Use your existing df
video_ids = df["video_id"].unique()
val_videos = set(np.random.choice(video_ids, size=int(0.2 * len(video_ids)), replace=False))

valid_count = 0
for idx, row in tqdm(df.iterrows(), total=len(df)):
    video_id = row["video_id"]
    frame_idx = int(row["timestamp"] * FPS)
    img_filename = f"img_{frame_idx:05d}.jpg"
    img_path = Path(FRAMES_DIR) / video_id / img_filename

    if not img_path.exists():
        continue

    # Read image to get dimensions
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    # Prepare bounding box (already normalized in CSV, but x1,x2 are fractions of image size)
    x1 = row["x1"]
    y1 = row["y1"]
    x2 = row["x2"]
    y2 = row["y2"]

    # Clamp to [0,1] just in case
    x1 = max(0.0, min(1.0, x1))
    x2 = max(0.0, min(1.0, x2))
    y1 = max(0.0, min(1.0, y1))
    y2 = max(0.0, min(1.0, y2))

    if x1 >= x2 or y1 >= y2:
        continue

    x_center = (x1 + x2) / 2.0
    y_center = (y1 + y2) / 2.0
    box_w = x2 - x1
    box_h = y2 - y1


    yolo_line = f"0 {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}\n"

    # Determine split
    split = "val" if video_id in val_videos else "train"

    # Copy image
    img_dest = Path(OUTPUT_DIR) / split / "images" / f"{video_id}_{frame_idx:05d}.jpg"
    cv2.imwrite(str(img_dest), img)

    # Write label
    label_dest = Path(OUTPUT_DIR) / split / "labels" / f"{video_id}_{frame_idx:05d}.txt"
    with open(label_dest, "w") as f:
        f.write(yolo_line)

    valid_count += 1

print(f"Created {valid_count} valid entries.")

100%|██████████| 34025/34025 [00:56<00:00, 603.32it/s]

Created 34022 valid entries.


In [11]:
label_dir = "/content/cow_detection_dataset/train/labels"
sample_label = os.listdir(label_dir)[0]
with open(os.path.join(label_dir, sample_label), "r") as f:
    print(f"Content of {sample_label}:")
    print(f.read())

Content of 488_00175.txt:
0 0.125000 0.087891 0.078125 0.105469



In [12]:
img_dir = "/content/cow_detection_dataset/train/images"
base = sample_label.replace(".txt", ".jpg")
print(f"Image exists: {os.path.exists(os.path.join(img_dir, base))}")

Image exists: True


In [13]:
print(df[["video_id", "timestamp", "x1", "y1", "x2", "y2"]].head(10))

  video_id  timestamp        x1        y1        x2        y2
0      618          2  0.457031  0.355469  0.621094  0.675781
1      618          2  0.457031  0.355469  0.621094  0.675781
2      618          2  0.789062  0.488281  0.894531  0.828125
3      618          2  0.789062  0.488281  0.894531  0.828125
4      618          2  0.210938  0.332031  0.335938  0.582031
5      618          2  0.058594  0.460938  0.164062  0.781250
6      618          2  0.132812  0.757812  0.574219  0.988281
7      618          2  0.132812  0.757812  0.574219  0.988281
8      618          2  0.031250  0.355469  0.113281  0.531250
9      618          2  0.031250  0.355469  0.113281  0.531250


In [14]:

video_id = "618"
frame_idx = 2 * 25  # timestamp 2
img_path = Path(FRAMES_DIR) / video_id / f"img_{frame_idx:05d}.jpg"
print(img_path.exists())  # Should be True

True


In [16]:
# Verify a label file is correct
sample_label = "/content/cow_detection_dataset/train/labels/618_00050.txt"  # adjust name
with open(sample_label, "r") as f:
    print(f.read())
# Should show numbers between 0 and 1, e.g., "0 0.5 0.5 0.2 0.2"

from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data="/content/cow_detection.yaml", epochs=100, batch=16, imgsz=640, cache=False, workers=2)

0 0.156250 0.326172 0.078125 0.113281

Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cow_detection.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimize

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c71ff5c5700>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [25]:
model = YOLO("/content/runs/detect/train/weights/last.pt")  # use your actual path
model.train(resume=True, epochs=50)

WARNING ⚠️ model '/content/runs/detect/train/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=3

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79579d38bb30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import shutil
from pathlib import Path

# Path to your training results
results_dir = Path("/content/runs")   # or "cow_detector/yolo11n_cbvd"

# Destination in Drive (create a folder for your cow project)
drive_dir = Path("/content/drive/MyDrive/cow_runs_result_100_epochs/")
drive_dir.mkdir(parents=True, exist_ok=True)

# Copy everything
shutil.copytree(results_dir, drive_dir / "yolo11_training_results", dirs_exist_ok=True)
print(f"Saved to {drive_dir / 'yolo11_training_results'}")

Saved to /content/drive/MyDrive/cow_runs_result_100_epochs/yolo11_training_results
